In [1]:
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.nn.functional import scaled_dot_product_attention
import math
import time
import os
from torch.cuda.amp import GradScaler
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.distributed as dist
from transformers import GPT2Tokenizer
from torch.utils.data import DataLoader

class LlamaRMSNorm(nn.Module):
    """
        LlamaRMSNorm is equivalent to T5LayerNorm.
        x / sqrt(mean(x^2, axis=-1) + eps) * weight
    """
    
    def __init__(self, d_model, eps=1e-5):
        """
        Args:
            d_model (int)  : Dimensionality of the input tensor.
            eps     (float): A small constant added to avoid division by zero.
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x): # return x / torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.weight * x / norm
    
class LlamaRotaryEmbedding(nn.Module):
    """
    # https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py#L240
    Rotary Positional Embedding (RoPE) for transformers Implemntation derived from https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py
    """
    def __init__(self, dim: int, theta: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.theta = theta

    def forward(self, x: torch.Tensor, seq_len: int) -> torch.Tensor:
       
        B, T, H, H_D = x.shape

        # Generate position indices
        position = torch.arange(T, dtype=torch.bfloat16, device=x.device).unsqueeze(-1)

        # Generate frequencies
        freqs = torch.exp(
            torch.arange(0, H_D, 2, dtype=torch.bfloat16, device=x.device) * 
            -(torch.log(torch.tensor(self.theta)) / H_D)
                                                                
        )

        # Compute sinusoids
        sinusoid = position * freqs
        sin = torch.sin(sinusoid)
        cos = torch.cos(sinusoid)

        # Reshape sin and cos to match the input tensor's shape
        sin = sin.unsqueeze(0).unsqueeze(2)  # Shape: (1, T, 1, D // 2)
        cos = cos.unsqueeze(0).unsqueeze(2)  # Shape: (1, T, 1, D // 2)

        # Apply rotary embeddings
        x_rotated = x.clone()
        x_rotated[..., 0::2] = x[..., 0::2] * cos - x[..., 1::2] * sin
        x_rotated[..., 1::2] = x[..., 1::2] * cos + x[..., 0::2] * sin

        return x_rotated
        
class LlamaSdpaAttention(nn.Module):
    """
    Implements multi-head self-attention with rotary embeddings and scaled dot-product attention.
    """
    
    def __init__(self, hidden_size, num_heads):
        """
        Args:
            hidden_size (int): Total dimensionality of the input embeddings.
            num_heads   (int): Number of attention heads.
        """
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.num_key_value_heads = 3

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, self.head_dim * self.num_key_value_heads, bias=False)
        self.v_proj = nn.Linear(hidden_size, self.head_dim * self.num_key_value_heads, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.rotary_emb = LlamaRotaryEmbedding(self.head_dim)

    def forward(self, x, attention_mask=None):
        """
        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].
            attention_mask (torch.Tensor, optional): Mask to control which positions are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: The output tensor after applying multi-head attention, of shape [batch_size, seq_len, hidden_size].
        """
        batch_size, seq_len, _ = x.shape
        B, T, C = x.size()

        # Project queries, keys and values
        q = self.q_proj(x).view(batch_size, seq_len, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim)
        v = self.v_proj(x).view(batch_size, seq_len, self.num_key_value_heads, self.head_dim)

        # Reshape for attention
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        q = self.rotary_emb(q, T)
        k = self.rotary_emb(k, T)

        # Repeat k/v heads if num_key_value_heads < num_attention_heads
        if self.num_key_value_heads != self.num_heads:
            k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1) # B,kv_head,T,D -> B,H,T,D
            v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=1) # B,kv_head,T,D -> B,H,T,D

        # Scaled dot product attention
        attn_output = scaled_dot_product_attention(
            q, k, v,
            attn_mask=attention_mask,
            dropout_p=0.0,
            is_causal=True
        )

        # Reshape and project output
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_size)
        return self.o_proj(attn_output)

class LlamaMLP(nn.Module):
    """
    Implements the MLP (Multilayer Perceptron) block used in LLaMA transformer layers.
    The MLP block applies a feed-forward network with non-linearity and projections.
    """
    
    def __init__(self, hidden_size, intermediate_size):
        """
        Initializes the MLP block with three linear layers and an activation function.

        Args:
            hidden_size (int): Dimensionality of the input and output embeddings.
            intermediate_size (int): Dimensionality of the intermediate feed-forward layer.
        """
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = nn.SiLU()

    def forward(self, x):
        """
        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].

        Returns:
            torch.Tensor: Output tensor of shape [batch_size, seq_len, hidden_size].
        """
        return self.down_proj(self.act_fn(self.gate_proj(x) + self.up_proj(x)))
    
class LlamaDecoderLayer(nn.Module):
    """
    Implements a single transformer decoder layer used in the LLaMA model. Each layer
    consists of a self-attention mechanism, followed by a feed-forward network (MLP),
    with residual connections and normalization applied at steps.
    """
    
    def __init__(self, hidden_size, num_heads, intermediate_size):
        """
        Args:
            hidden_size (int): Dimensionality of input and output embeddings.
            num_heads (int): Number of attention heads for multi-head self-attention.
            intermediate_size (int): Dimensionality of the feed-forward network.
        """
        super().__init__()
        self.attn = LlamaSdpaAttention(hidden_size, num_heads)
        self.input_layernorm = LlamaRMSNorm(hidden_size)
        self.post_attention_layernorm = LlamaRMSNorm(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)
    
    def forward(self, x, attention_mask=None):
        """
        Forward pass for a single decoder layer.

        Args:
            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, hidden_size].
            attention_mask (torch.Tensor, optional): Mask for controlling which positions to attend to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: Output tensor of shape [batch_size, seq_len, hidden_size].
        """
        # Self-attention
        residual = x
        x = self.attn(self.input_layernorm(x), attention_mask)
        x = x + residual

        # Feedforward
        residual = x
        x = self.mlp(self.post_attention_layernorm(x))
        x = x + residual

        return x
    
class LlamaModel(nn.Module):
    """
    Implements the LLaMA transformer model, consisting of an embedding layer,
    a stack of decoder layers, and an output normalization layer.
    """
    
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size):
        """
        Args:
            vocab_size (int): Size of the input vocabulary.
            hidden_size (int): Dimensionality of the input embeddings and hidden states.
            num_hidden_layers (int): Number of stacked decoder layers in the model.
            num_heads (int): Number of attention heads in each decoder layer.
            intermediate_size (int): Dimensionality of the feedforward layers in the MLPs.
        """
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            LlamaDecoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_hidden_layers)
        ])
        self.norm = LlamaRMSNorm(hidden_size)
    
    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids (torch.Tensor): Input tensor of token IDs of shape [batch_size, seq_len].
            attention_mask (torch.Tensor, optional): Attention mask to control which tokens are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: The final hidden states of shape [batch_size, seq_len, hidden_size].
        """
        x = self.embed_tokens(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask)

        return self.norm(x)
    
class LlamaForCausalLM(nn.Module):
    """
    LLaMA model for causal language modeling. It integrates the
    transformer model (`LlamaModel`) with an additional output head for predicting 
    the next token in a sequence (causal language modeling task). The model shares weights 
    between the token embeddings and the output logits layer (lm_head).
    """
    
    def __init__(self, vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len):
        """
        Args:
            vocab_size (int): The size of the input vocabulary.
            hidden_size (int): Dimensionality of the input embeddings and hidden states.
            num_hidden_layers (int): Number of stacked decoder layers.
            num_heads (int): Number of attention heads for each decoder layer.
            intermediate_size (int): Size of the intermediate layer in the MLPs.
            max_seq_len (int): Maximum sequence length.
        """
        super().__init__()
        self.model = LlamaModel(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        self.lm_head.weight = self.model.embed_tokens.weight # Weight Sharing

    def forward(self, input_ids, attention_mask=None):
        """
        Args:
            input_ids (torch.Tensor): Input tensor of token IDs with shape [batch_size, seq_len].
            attention_mask (torch.Tensor, optional): Attention mask to control which tokens are attended to.
                                                     Shape [batch_size, seq_len].

        Returns:
            torch.Tensor: Logits for each token in the vocabulary with shape [batch_size, seq_len, vocab_size].
        """
        x = self.model(input_ids, attention_mask)
        logits = self.lm_head(x)
        return logits
    
    def resize_token_embeddings(self, new_num_tokens):
        """
        Args:
            new_num_tokens (int): The new number of tokens in the vocabulary.

        Returns:
            torch.nn.Embedding: The new token embedding layer with the updated vocabulary size.
        """
        # Resize the token embeddings
        old_embeddings = self.model.embed_tokens
        new_emmbeddings = nn.Embedding(new_num_tokens, old_embeddings.embedding_dim)

        # Copy the exisiting embeddings
        num_tokens_to_copy = min(old_embeddings.num_embeddings, new_num_tokens)
        new_emmbeddings.weight.data[:num_tokens_to_copy, :] = old_embeddings.weight.data[:num_tokens_to_copy, :]

        # Replace the old embeddings
        self.model.embed_tokens = new_emmbeddings

        # Update the lm_head to match the new embedding size
        old_lm_head = self.lm_head
        self.lm_head = nn.Linear(old_lm_head.in_features, new_num_tokens, bias=False)

        # Copy the exisiting weights
        num_tokens_to_copy = min(old_lm_head.out_features, new_num_tokens)
        self.lm_head.weight.data[:num_tokens_to_copy, :] = old_lm_head.weight.data[:num_tokens_to_copy, :]

        # Maintain weight sharing
        self.lm_head.weight = self.model.embed_tokens.weight

        return self.model.embed_tokens
    
# First load the tokenizer and set pad token
tokenizer = GPT2Tokenizer.from_pretrained("HuggingFaceTB/cosmo2-tokenizer")
tokenizer.pad_token = tokenizer.eos_token
model_vocab_size = len(tokenizer)

# Now create model with the correct vocab size from the start
vocab_size = model_vocab_size
hidden_size = 576
num_hidden_layers = 30
num_heads = 9
intermediate_size = 1536
max_seq_len = 2048

# Instantiate the model with correct vocab size
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model = LlamaForCausalLM(vocab_size, hidden_size, num_hidden_layers, num_heads, intermediate_size, max_seq_len)

# Print model size
model_size = sum(p.numel() for p in model.parameters())
print(f"Model Size: {model_size / 1e6:.2f}M parameters.")

# Load Cosmopedia Dataset
def load_cosmopedia_dataset(batch_size, seq_length):
    try:
        dataset = load_dataset(
            "HuggingFaceTB/smollm-corpus",
            name="cosmopedia-v2",
            split="train",
            streaming=True
        )

        def encode(example):
            tokens = tokenizer(
                example["text"],
                truncation=True,
                max_length = seq_length,
                return_tensors="pt",
                padding="max_length",
            )
            # Use clone().detach() as recommended
            input_ids = tokens['input_ids'].squeeze(0).clone().detach()
            input_ids = torch.clamp(input_ids, max=vocab_size - 1)
            return {
                'input_ids': input_ids,
                'labels': input_ids.clone().detach()
            }
        
        dataset = dataset.map(encode, remove_columns=['text'], batched=False)
        dataset = dataset.with_format("torch")
        dataloader = DataLoader(dataset, batch_size=batch_size)
        return dataloader
    except Exception as e:
        print(f"Error loading dataset: {str(e)}")
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
        return load_wikitext_dataset(dataset, batch_size, seq_length)
    
def load_wikitext_dataset(dataset, batch_size, seq_length):
    def encode(example):
        tokens = tokenizer(
            example['text'],
            truncation=True,
            max_length = seq_length,
            return_tensors="pt",
            padding="max_length",
        )
        # Use clone().detach() as recommended
        input_ids = tokens['input_ids'].squeeze(0).clone().detach()
        input_ids = torch.clamp(input_ids, max=vocab_size - 1)
        return {
            'input_ids': input_ids,
            'labels': input_ids.clone().detach()
        }
    
    dataset = dataset.map(encode, remove_columns=['text'], batched=False)
    dataset = dataset.with_format("torch")
    dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=10)
    return dataloader

print("Loading Dataset ...")
train_loader = load_cosmopedia_dataset(batch_size=16, seq_length=1024)

print("\nSample texts from dataset:")
sample_texts = []
sample_count = 3
for i, batch in enumerate(train_loader):
    if i >= sample_count:
        break

    sample_idx = torch.randint(0, batch['input_ids'].size(0), (1,)).item()
    decoded_text = tokenizer.decode(batch['input_ids'][sample_idx], skip_special_tokens=True)
    # Get first 10 words as prompt
    prompt = ' '.join(decoded_text.split()[:10])
    sample_texts.append({
        'full_text': decoded_text[:200] + " ... ",
        'prompt': prompt
    })
    print(f"\nSample {i+1}:")
    print(f"Full text: {sample_texts[-1]['full_text']}")
    print(f"Prompt: {sample_texts[-1]['prompt']}")


torch.set_float32_matmul_precision('high')

# Training setup
def setup_training(model, batch_size=16):
    """"Hi"""
    # Enable cudnn benchmarking and deterministic mode
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

    # Move the model to GPU
    device = torch.device("cuda:1")
    model = model.to(device)

    # Enable gradient scaler for mixed precision training
    scaler = GradScaler()

    # Optimize with larger batch size and lr
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        betas=(0.9, 0.95),
        eps=1e-8,
        weight_decay=0.1
    )

    # Learning rate scheduler with longer warmup
    lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=3e-4,
        epochs=1,
        steps_per_epoch=10000,
        pct_start=0.1,
        anneal_strategy='cos'
    )

    return device, optimizer, lr_scheduler, scaler


def save_checkpoint(model, step, save_dir="checkpoints"):
    """
    Save only the model's state_dict to a checkpoint file, using the step in the filename.
    """
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Save only the model's state_dict
    checkpoint = model.state_dict()

    path = os.path.join(save_dir, f'model_step_{step}.pt')
    torch.save(checkpoint, path)
    print(f"Checkpoint saved: {path}")

def load_checkpoint(model, chkpt_path, device):
    """
    Load only the model's state_dict from a checkpoint file.
    """
    if os.path.exists(chkpt_path):
        checkpoint = torch.load(chkpt_path, map_location=device)
        model.load_state_dict(checkpoint)
        print(f"Checkpoint loaded: {chkpt_path}")
    else:
        raise FileNotFoundError(f"Checkpoint file not found: {chkpt_path}")


# Training metrics tracking
best_loss = float('inf')
running_loss = 0.0
log_interval = 1

# Setup training optimizations
device, optimizer, lr_scheduler, scaler = setup_training(model)

# Use the same batch size
train_loader = load_cosmopedia_dataset(batch_size=8, seq_length=1024)

# Add gradient accumulation
gradient_accumulation_steps = 8

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id, reduction="mean")

# Training loop
print("Starting training...")
model.train()
num_epochs = 1
max_steps = 5000
running_loss = 0.0
optimizer.zero_grad()

# Checkpoint path and resume logic
resume_training = False
chkpt_path = "checkpoints/model_step_500.pt"

if resume_training and os.path.exists(chkpt_path):
    step = load_checkpoint(model, chkpt_path, device)
else:
    step = 0

for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} of {num_epochs}")
    for batch_idx, batch in enumerate(train_loader):
        if step >= max_steps:
            print("Reached max steps. Ending training.")
            break

        if batch_idx % gradient_accumulation_steps == 0:
            accum_start_time = time.time()

        t0 = time.time()

        input_ids, labels = batch['input_ids'], batch['labels']
        input_ids, labels = input_ids.to(device), labels.to(device)

        # AMP
        with torch.autocast(device_type="cuda:1", dtype=torch.bfloat16):
            outputs = model(input_ids)
            shift_logits = outputs[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()  # Use shifted labels for causal LM
            loss = criterion(shift_logits.view(-1, vocab_size), shift_labels.view(-1))
            loss = loss / gradient_accumulation_steps  # Accumulate loss over multiple steps

        # Backward pass
        scaler.scale(loss).backward()  # Use GradScaler for better stability with mixed precision

        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)  # Apply gradients
            scaler.update()  # Update the scaler for mixed precision
            optimizer.zero_grad()  # Reset gradients
            lr_scheduler.step()
            torch.cuda.synchronize()  # Ensure proper timing and synchronization
            step += 1

            # Update metrics
            running_loss += loss.item() * gradient_accumulation_steps

            # Timing and logging for complete accumulated steps
            accum_end_time = time.time()
            total_step_time = (accum_end_time - accum_start_time) * 1000
            batch_tokens = input_ids.size(0) * input_ids.size(1)
            total_tokens = batch_tokens * gradient_accumulation_steps
            tokens_per_sec = total_tokens / (accum_end_time - accum_start_time)

            if step % log_interval == 0:
                avg_loss = running_loss / log_interval
                print(f"Step {step} | Loss: {avg_loss:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.6f} | "
                      f"Total Step Time: {total_step_time:.2f}ms | Tokens/sec: {tokens_per_sec:.2f}")
                running_loss = 0.0

            # Save checkpoint every 5000 steps
            # if step % 5000 == 0:
            #     save_checkpoint(model, step)

            # Generate sample text every 500 steps
            if step % 500 == 0:
                model.eval()
                with torch.no_grad():
                    for i, sample in enumerate(sample_texts):
                        print(f"\nGeneration {i+1} with prompt: {sample['prompt']}")
                        inputs = tokenizer(sample['prompt'], return_tensors="pt", padding=True, truncation=True).to(device)
                        outputs = model(inputs["input_ids"])
                        prediction = torch.argmax(outputs, dim=-1)
                        decoded = tokenizer.decode(prediction[0], skip_special_tokens=True)
                        print(f"Generated: {decoded}\n")
                model.train()

    if step >= max_steps:
        break

# Save final checkpoint
save_checkpoint(model, step)

Model Size: 134.52M parameters.
Loading Dataset ...


Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/104 [00:00<?, ?it/s]


Sample texts from dataset:


/home/jovyan/env/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


...

Starting training...
Epoch 1 of 1


/home/jovyan/env/lib/python3.10/site-packages/datasets/formatting/torch_formatter.py:87: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(value, **{**default_dtype, **self.torch_tensor_kwargs})


Step 1 | Loss: 291.0684 | LR: 0.000012 | Total Step Time: 1684.86ms | Tokens/sec: 38897.07
Step 2 | Loss: 286.0152 | LR: 0.000012 | Total Step Time: 1102.13ms | Tokens/sec: 59463.18
Step 3 | Loss: 283.0332 | LR: 0.000012 | Total Step Time: 1073.05ms | Tokens/sec: 61074.34
Step 4 | Loss: 277.3873 | LR: 0.000012 | Total Step Time: 1108.40ms | Tokens/sec: 59126.85
Step 5 | Loss: 271.0402 | LR: 0.000012 | Total Step Time: 1056.99ms | Tokens/sec: 62002.67
Step 6 | Loss: 265.8299 | LR: 0.000012 | Total Step Time: 1084.04ms | Tokens/sec: 60455.51
Step 7 | Loss: 261.4918 | LR: 0.000012 | Total Step Time: 1137.51ms | Tokens/sec: 57613.64
Step 8 | Loss: 254.3843 | LR: 0.000012 | Total Step Time: 1062.26ms | Tokens/sec: 61694.62
Step 9 | Loss: 247.2551 | LR: 0.000012 | Total Step Time: 1071.49ms | Tokens/sec: 61163.55
Step 10 | Loss: 242.2403 | LR: 0.000012 | Total Step Time: 1040.36ms | Tokens/sec: 62993.61
Step 11 | Loss: 235.6064 | LR: 0.000012 | Total Step Time: 1056.23ms | Tokens/sec: 62047.

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step 500 | Loss: 33.7235 | LR: 0.000156 | Total Step Time: 1027.58ms | Tokens/sec: 63777.32

Generation 1 with prompt: Title: How to Make a Secret Shake Introduction: Creating a
Generated: ,
. the. for....
 Creating 


Generation 2 with prompt: ## Factorial Numbers and Factoradic Representation In mathematics, factorial of
Generated:  to... the..... and.. the


Generation 3 with prompt: Course Unit: Discovering the Roots of the Violin Introduction: Have
Generated: .:
. of, the of., to
.

Step 501 | Loss: 32.8528 | LR: 0.000157 | Total Step Time: 1023.13ms | Tokens/sec: 64054.54
Step 502 | Loss: 32.0207 | LR: 0.000157 | Total Step Time: 1006.09ms | Tokens/sec: 65139.15
Step 503 | Loss: 32.6084 | LR: 0.000158 | Total Step Time: 1026.34ms | Tokens/sec: 63853.98
Step 504 | Loss: 32.9534 | LR: 0.000158 | Total Step Time: 1025.54ms | Tokens/sec: 63903.98
Step 505 | Loss: 33.3855 | LR: 0.000158 | Total Step Time: 1064.00ms | Tokens/sec: 61593.77
Step 506 | Loss: 33.3031 | LR: 0.000159 | Total